In [20]:
import pandas as pd
import numpy as np

from IPython.display import display

from scipy.special import softmax
from scipy.stats import gmean

from xgboost import XGBClassifier
from sklearn.base import BaseEstimator,RegressorMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score , roc_auc_score , classification_report , confusion_matrix
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from skorch import NeuralNetBinaryClassifier

import time

In [21]:
df_train = pd.read_csv('/home/kyburg/kaggle/input/predicting_stellar_class/train.csv')
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 577347 entries, 0 to 577346
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 577347 non-null  int64  
 1   alpha              577347 non-null  float64
 2   delta              577347 non-null  float64
 3   u                  577347 non-null  float64
 4   g                  577347 non-null  float64
 5   r                  577347 non-null  float64
 6   i                  577347 non-null  float64
 7   z                  577347 non-null  float64
 8   redshift           577347 non-null  float64
 9   spectral_type      577347 non-null  str    
 10  galaxy_population  577347 non-null  str    
 11  class              577347 non-null  str    
dtypes: float64(8), int64(1), str(3)
memory usage: 62.9 MB


In [22]:
df_reduced = df_train.sample(frac=0.01)
y_class = df_reduced[['class']].copy()
le = LabelEncoder()
y_labels = le.fit_transform(y_class.values.ravel())
ohe = OneHotEncoder(sparse_output=False).set_output(transform='pandas')
y = ohe.fit_transform(y_labels.reshape(-1, 1))
X = df_reduced.drop(columns=['class', 'id'])
df_reduced.info()

<class 'pandas.DataFrame'>
Index: 5773 entries, 243170 to 212661
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5773 non-null   int64  
 1   alpha              5773 non-null   float64
 2   delta              5773 non-null   float64
 3   u                  5773 non-null   float64
 4   g                  5773 non-null   float64
 5   r                  5773 non-null   float64
 6   i                  5773 non-null   float64
 7   z                  5773 non-null   float64
 8   redshift           5773 non-null   float64
 9   spectral_type      5773 non-null   str    
 10  galaxy_population  5773 non-null   str    
 11  class              5773 non-null   str    
dtypes: float64(8), int64(1), str(3)
memory usage: 690.8 KB


In [23]:
def add_color_features(df):
    df = df.copy()
    df['u-g'] = df['u'] - df['g']
    df['g-r'] = df['g'] - df['r']
    df['r-i'] = df['r'] - df['i']
    df['i-z'] = df['i'] - df['z']

    df[['u_softmax', 'g_softmax', 'r_softmax', 'i_softmax', 'z_softmax']] = softmax(df[['u', 'g', 'r', 'i', 'z']], axis=1)
    lambda_coef = np.sqrt([365,475,658,806,900])
    df['lambda'] = lambda_coef[0]*df['u_softmax'] + lambda_coef[1]*df['g_softmax'] + lambda_coef[2]*df['r_softmax'] + lambda_coef[3]*df['i_softmax'] + lambda_coef[4]*df['z_softmax']
    df['K'] = 1/np.exp(df['lambda'])
    
    return df

In [24]:
def add_brightness_features(df):
    df = df.copy()
    color_columns = ['u','g','r','i','z']
    df['brightness'] = df[color_columns].sum(axis=1)

    df[['u_norm', 'g_norm', 'r_norm', 'i_norm', 'z_norm']] = df[color_columns].div(df['brightness'], axis=0)
    df['luminosity'] = np.square(df['redshift'])*df['brightness']
    
    return df

In [25]:
def add_log_features(df):
    df = df.copy()

    df[['u_log','g_log','r_log','i_log','z_log']] = np.log1p(df[['u','g','r','i','z']].clip(lower=0))
    df['redshift_log'] = np.log1p(df['redshift'].clip(lower=0))

    return df

In [26]:
def add_geometric_features(df):
    df = df.copy()

    if 'alpha' in df.columns:
        df['alpha_sin'] = np.sin(np.radians(df['alpha']))
        df['alpha_cos'] = np.cos(np.radians(df['alpha']))

        df.drop(columns='alpha', inplace=True)
    else:
        if ('alpha_sin' not in df.columns) or ('alpha_cos' not in df.columns):
            print(f'Function: add_geometric_features\nUnexpected columns: {df.columns}')

    return df

In [27]:
def trandform_categorical(df):
    df = df.copy()

    spectral_type_map = {'M' : 0, 'G/K' : 1, 'A/F' : 2, 'O/B' : 3}
    df['spectral_type'] = df['spectral_type'].map(spectral_type_map)
    df['galaxy_population'] = df['galaxy_population'].astype('category').cat.codes
    
    return df

In [28]:
def preprocessing(df):
    return add_log_features(add_geometric_features(add_color_features(add_brightness_features(trandform_categorical(df)))))

In [29]:
X = preprocessing(X)
scaler = StandardScaler().set_output(transform='pandas')
X = scaler.fit_transform(X)

In [30]:
X.info()

<class 'pandas.DataFrame'>
Index: 5773 entries, 243170 to 212661
Data columns (total 35 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   delta              5773 non-null   float64
 1   u                  5773 non-null   float64
 2   g                  5773 non-null   float64
 3   r                  5773 non-null   float64
 4   i                  5773 non-null   float64
 5   z                  5773 non-null   float64
 6   redshift           5773 non-null   float64
 7   spectral_type      5773 non-null   float64
 8   galaxy_population  5773 non-null   float64
 9   brightness         5773 non-null   float64
 10  u_norm             5773 non-null   float64
 11  g_norm             5773 non-null   float64
 12  r_norm             5773 non-null   float64
 13  i_norm             5773 non-null   float64
 14  z_norm             5773 non-null   float64
 15  luminosity         5773 non-null   float64
 16  u-g                5773 non-null 

In [31]:
class NNClassifier(nn.Module):
    def __init__(self, topology, alpha=1.0):
        super().__init__()
        self.layer1 = nn.Linear(35, topology[0])
        self.norm1 = nn.BatchNorm1d(topology[0])
        self.activation1 = nn.ELU(alpha=alpha)
        self.layer2 = nn.Linear(topology[0], topology[1])
        self.activation2 = nn.ELU(alpha=alpha)
        self.layer3 = nn.Linear(topology[1], topology[2])
        self.activation3 = nn.ELU(alpha=alpha)
        self.logits = nn.Linear(topology[2], 3)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.norm1(x)
        x = self.activation1(x)
        x = self.layer2(x)
        x = self.activation2(x)
        x = self.layer3(x)
        x = self.activation3(x)
        x = self.logits(x)
        return x

In [32]:
class MyDataset(Dataset):
    def __init__(self, features, targets, weights=None):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets.values, dtype=torch.float32)
        if weights is not None:
            self.weights = torch.tensor(weights, dtype=torch.float32)
        else:
            self.weights = torch.zeros(features.shape[0])
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx]


- We train a model of each kind to build a Voting Classifier. Then, AdaBoost is used with 5 Voting models as base models
- We train a Stacking model trained with 5 model of each kind. The meta model is either Logistic or NeuralNetwork (2 layers)
- We train a VotingClassifier with 5 models of each kind

In [33]:
class MyVotingClassifier(BaseEstimator,RegressorMixin):
    def __init__(self, models, ohe):
        self.models = models
        self.ohe = ohe
    def fit(self, X, y, sample_weight=None):
        for (model,training_parameters) in self.models:
            if 'NN' == training_parameters['model_name']:
                y_ohe = ohe.transform(y.reshape(-1,1))
                train_loader = DataLoader(dataset=MyDataset(X,y_ohe, sample_weight), batch_size=training_parameters['batch_size'])
                criterion=nn.CrossEntropyLoss()
                optimizer=torch.optim.Adam(model.parameters(), lr=training_parameters['lr'])
                for epoch in range(training_parameters['max_epochs']):
                    for x_batch,y_batch,weight_batch in train_loader:
                        optimizer.zero_grad()
                        y_pred = model(x_batch)
                        loss = criterion(y_pred, y_batch)
                        if sample_weight is not None:
                            loss = (loss*weight_batch).mean()
                        loss.backward()
                        optimizer.step()
            else:
                model.fit(X,y, sample_weight)
            print(f'Model with name: {training_parameters['model_name']}, has been fitted')
        self.classes_ = np.array(np.unique(y))
        return self
    def predict(self, X):
        outputs = []
        for (model,training_parameters) in self.models:
            if 'NN' == training_parameters['model_name']:
                X_torch = torch.from_numpy(X.astype(np.float32))
                with torch.no_grad():
                    output = model(X_torch).numpy()
                outputs += [output]
            else:
                output = model.predict_proba(X)
                outputs += [output]
        outputs = np.array(outputs)
        votes = outputs.mean(axis=0)
        return np.argmax(votes, axis=1)

In [34]:
class MyStackingModel(BaseEstimator,RegressorMixin):
    def __init__(self, models, ohe, nn_meta_estimator, meta_estimator, meta_estimator_parameters):
        self.models = models
        self.ohe = ohe
        self.nn_meta_estimator = nn_meta_estimator
        self.meta_estimator = meta_estimator
        self.meta_estimator_parameters = meta_estimator_parameters
    def fit(self, X, y):
        y_ohe = ohe.transform(y.reshape(-1,1))
        for (model,training_parameters) in self.models:
            if 'NN' == training_parameters['model_name']:
                train_loader = DataLoader(dataset=MyDataset(X.values,y_ohe), batch_size=training_parameters['batch_size'])
                criterion=nn.CrossEntropyLoss()
                optimizer=torch.optim.Adam(model.parameters(), lr=training_parameters['lr'])
                for epoch in range(training_parameters['max_epochs']):
                    for x_batch,y_batch,_ in train_loader:
                        optimizer.zero_grad()
                        y_pred = model(x_batch)
                        loss = criterion(y_pred, y_batch)
                        loss.backward()
                        optimizer.step()
            else:
                model.fit(X,y)
            print(f'Model with name: {training_parameters['model_name']}, has been fitted')
        # Train the meta estimator using the logits of the input
        outputs = []
        for (model,training_parameters) in self.models:
            if 'NN' == training_parameters['model_name']:
                X_torch = torch.from_numpy(X.values.astype(np.float32))
                with torch.no_grad():
                    output = model(X_torch)
                    output = nn.Softmax(dim=1)(output)
                outputs += [output.numpy()]
            else:
                output = model.predict_proba(X)
                outputs += [output]
        outputs = np.array(outputs)           # models * examples * 3
        outputs = np.moveaxis(outputs, 1, 0)
        outputs = outputs.reshape(outputs.shape[0], outputs.shape[1] * outputs.shape[2])
        if self.nn_meta_estimator:
            train_loader = DataLoader(dataset=MyDataset(outputs,y_ohe), batch_size=self.meta_estimator_parameters['batch_size'])
            criterion=nn.CrossEntropyLoss()
            optimizer=torch.optim.Adam(self.meta_estimator.parameters(), lr=self.meta_estimator_parameters['lr'])
            for epoch in range(self.meta_estimator_parameters['max_epochs']):
                for x_batch,y_batch,_ in train_loader:
                    optimizer.zero_grad()
                    y_pred = self.meta_estimator(x_batch)
                    loss = criterion(y_pred, y_batch)
                    loss.backward()
                    optimizer.step()
        else:
            self.meta_estimator.fit(outputs, y)
        return self
    def predict(self, X):
        outputs = []
        for (model,training_parameters) in self.models:
            if 'NN' == training_parameters['model_name']:
                X_torch = torch.from_numpy(X.values.astype(np.float32))
                with torch.no_grad():
                    output = model(X_torch)
                    output = nn.Softmax(dim=1)(output)
                outputs += [output.numpy()]
            else:
                output = model.predict_proba(X)
                outputs += [output]
        outputs = np.array(outputs)
        outputs = np.moveaxis(outputs, 1, 0)
        outputs = outputs.reshape(outputs.shape[0], outputs.shape[1] * outputs.shape[2])
        if self.nn_meta_estimator:
            with torch.no_grad():
                outputs_torch = torch.from_numpy(outputs.astype(np.float32))
                out = self.meta_estimator(outputs_torch)
            return torch.argmax(out, dim=1).numpy()
        else:
            return self.meta_estimator.predict(outputs)

In [35]:
def get_models(model_params):
    models = []
    for i in range(len(model_params)):
        training_params = {'model_name' : model_params[i]['model_name']}
        if 'Logistic' == model_params[i]['model_name']:
            clf = LogisticRegression(max_iter=model_params[i]['max_iter'], class_weight='balanced', solver=model_params[i]['solver'], C=model_params[i]['C'])
        if 'RandomForest' == model_params[i]['model_name']:
            clf = RandomForestClassifier(n_estimators=model_params[i]['n_estimators'], max_depth=model_params[i]['max_depth'], min_samples_split=model_params[i]['min_samples_split'], class_weight='balanced', n_jobs=-1)
        if 'XGBClassifier' == model_params[i]['model_name']:
            clf = XGBClassifier(n_estimators=model_params[i]['n_estimators'], learning_rate=model_params[i]['learning_rate'], max_depth=model_params[i]['max_depth'], subsample=model_params[i]['subsample'], colsample_bytree=model_params[i]['colsample_bytree'], eval_metric='logloss', verbosity=0, n_jobs=-1)
        if 'LightGBM' == model_params[i]['model_name']:
            clf = LGBMClassifier(n_estimators=model_params[i]['n_estimators'], learning_rate=model_params[i]['learning_rate'], max_depth=model_params[i]['max_depth'], subsample=model_params[i]['subsample'], colsample_bytree=model_params[i]['colsample_bytree'], verbosity=-1, n_jobs=-1)
        if 'AdaBoost' == model_params[i]['model_name']:
            clf = AdaBoostClassifier(n_estimators=model_params[i]['n_estimators'], learning_rate=model_params[i]['learning_rate'])
        if 'NN' == model_params[i]['model_name']:
            clf = NNClassifier(topology=model_params[i]['topology'], alpha=model_params[i]['alpha'])
            training_params['batch_size'] = model_params[i]['batch_size']
            training_params['lr'] = model_params[i]['lr']
            training_params['max_epochs'] = model_params[i]['max_epochs']
        models += [(clf, training_params)]
    return models

In [36]:
models_1 = get_models([
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cholesky', 'C' : 5.0},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 25, 'min_samples_split' : 10},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.7, 'colsample_bytree' : 0.7},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 9, 'subsample' : 0.8, 'colsample_bytree' : 0.7},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'NN', 'topology' : [48, 24, 12], 'alpha' : 0.1, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100}
])
models_5 = get_models([
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cholesky', 'C' : 5.0},
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cg', 'C' : 5.0},
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'lbfgs', 'C' : 5.0},
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cholesky', 'C' : 2.0},
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cg', 'C' : 2.0},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 25, 'min_samples_split' : 10},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 25, 'min_samples_split' : 10},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 25, 'min_samples_split' : 10},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 50, 'min_samples_split' : 10},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 50, 'min_samples_split' : 10},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.7, 'colsample_bytree' : 0.7},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 9, 'subsample' : 0.7, 'colsample_bytree' : 0.9},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 25, 'subsample' : 0.9, 'colsample_bytree' : 0.9},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 9, 'subsample' : 0.7, 'colsample_bytree' : 0.8},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.9, 'colsample_bytree' : 0.8},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 9, 'subsample' : 0.8, 'colsample_bytree' : 0.7},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 15, 'subsample' : 0.7, 'colsample_bytree' : 0.8},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 25, 'subsample' : 0.8, 'colsample_bytree' : 0.8},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.8, 'colsample_bytree' : 0.7},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.7, 'colsample_bytree' : 0.7},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'NN', 'topology' : [48, 24, 12], 'alpha' : 0.1, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100},
    {'model_name' : 'NN', 'topology' : [48, 24, 12], 'alpha' : 0.3, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100},
    {'model_name' : 'NN', 'topology' : [48, 24, 12], 'alpha' : 1.0, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100},
    {'model_name' : 'NN', 'topology' : [48, 24, 6], 'alpha' : 0.1, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100},
    {'model_name' : 'NN', 'topology' : [48, 24, 6], 'alpha' : 0.3, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100}
])

In [37]:
logistic_ensemble = MyStackingModel(models_5, ohe, False, LogisticRegression(), {'model_name' : 'logistic'})
cv_scores_logistic = cross_val_score(logistic_ensemble, X, y_labels, cv=skf)
cv_scores_logistic


Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: AdaBoost, has been fitted
Model wit

array([0.72396134, 0.69683615, 0.72787522, 0.72616233, 0.72295955])

In [38]:
ada_boost_ensemble = AdaBoostClassifier(estimator=MyVotingClassifier(models_1, ohe), n_estimators=5)
cv_scores_ada = cross_val_score(ada_boost_ensemble, X.values, y_labels, cv=skf)
cv_scores_ada

Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/xgboost/core.py:748: FutureWarning: Pass `sample_weight` as keyword args.
  warnings.warn(msg, FutureWarning)


Model with name: XGBClassifier, has been fitted
Model with name: LightGBM, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: NN, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted


...

Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/xgboost/core.py:748: FutureWarning: Pass `sample_weight` as keyword args.
  warnings.warn(msg, FutureWarning)


Model with name: XGBClassifier, has been fitted
Model with name: LightGBM, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: NN, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid featur

array([0.94458874, 0.95584416, 0.94458874, 0.95233969, 0.95407279])

In [39]:
voting_ensemble = MyVotingClassifier(models_5, ohe)
cv_scores_voting = cross_val_score(voting_ensemble, X.values, y_labels, cv=skf)
cv_scores_voting

Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/xgboost/core.py:748: FutureWarning: Pass `sample_weight` as keyword args.
  warnings.warn(msg, FutureWarning)


Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: NN, has been fitted
Model with name: NN, has been fitted
Model with name: NN, has been fitted
Model with name: NN, has been fitted
Model with name: NN, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid featur

Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted


...

Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted
Model with name: RandomForest, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/xgboost/core.py:748: FutureWarning: Pass `sample_weight` as keyword args.
  warnings.warn(msg, FutureWarning)


Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: XGBClassifier, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: LightGBM, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: NN, has been fitted
Model with name: NN, has been fitted
Model with name: NN, has been fitted
Model with name: NN, has been fitted
Model with name: NN, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid featur

array([0.67130657, 0.71119655, 0.78069944, 0.745379  , 0.72776372])

In [40]:
models_1 = get_models([
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cholesky', 'C' : 5.0},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 25, 'min_samples_split' : 10},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.7, 'colsample_bytree' : 0.7},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 9, 'subsample' : 0.8, 'colsample_bytree' : 0.7},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'NN', 'topology' : [48, 24, 12], 'alpha' : 0.1, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100}
])
models_5 = get_models([
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cholesky', 'C' : 5.0},
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cg', 'C' : 5.0},
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'lbfgs', 'C' : 5.0},
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cholesky', 'C' : 2.0},
    {'model_name' : 'Logistic', 'max_iter' : 10000, 'solver' : 'newton-cg', 'C' : 2.0},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 25, 'min_samples_split' : 10},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 25, 'min_samples_split' : 10},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 25, 'min_samples_split' : 10},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 50, 'min_samples_split' : 10},
    {'model_name' : 'RandomForest', 'n_estimators' : 1000, 'max_depth' : 50, 'min_samples_split' : 10},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.7, 'colsample_bytree' : 0.7},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 9, 'subsample' : 0.7, 'colsample_bytree' : 0.9},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 25, 'subsample' : 0.9, 'colsample_bytree' : 0.9},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 9, 'subsample' : 0.7, 'colsample_bytree' : 0.8},
    {'model_name' : 'XGBClassifier', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.9, 'colsample_bytree' : 0.8},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 9, 'subsample' : 0.8, 'colsample_bytree' : 0.7},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 15, 'subsample' : 0.7, 'colsample_bytree' : 0.8},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 25, 'subsample' : 0.8, 'colsample_bytree' : 0.8},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.8, 'colsample_bytree' : 0.7},
    {'model_name' : 'LightGBM', 'n_estimators' : 1000, 'learning_rate' : 0.1, 'max_depth' : 7, 'subsample' : 0.7, 'colsample_bytree' : 0.7},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'AdaBoost', 'n_estimators' : 1000, 'learning_rate' : 0.1},
    {'model_name' : 'NN', 'topology' : [48, 24, 12], 'alpha' : 0.1, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100},
    {'model_name' : 'NN', 'topology' : [48, 24, 12], 'alpha' : 0.3, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100},
    {'model_name' : 'NN', 'topology' : [48, 24, 12], 'alpha' : 1.0, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100},
    {'model_name' : 'NN', 'topology' : [48, 24, 6], 'alpha' : 0.1, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100},
    {'model_name' : 'NN', 'topology' : [48, 24, 6], 'alpha' : 0.3, 'batch_size' : 32, 'lr' : 1e-4, 'max_epochs': 100}
])

In [49]:
#df_reduced = df_train.sample(frac=1)
y_class = df_train[['class']].copy()
le = LabelEncoder()
y_labels = le.fit_transform(y_class.values.ravel())
ohe = OneHotEncoder(sparse_output=False).set_output(transform='pandas')
y = ohe.fit_transform(y_labels.reshape(-1, 1))
X = df_train.drop(columns=['class', 'id'])
X = preprocessing(X)
scaler = StandardScaler().set_output(transform='pandas')
X = scaler.fit_transform(X)
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 577347 entries, 0 to 577346
Data columns (total 35 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   delta              577347 non-null  float64
 1   u                  577347 non-null  float64
 2   g                  577347 non-null  float64
 3   r                  577347 non-null  float64
 4   i                  577347 non-null  float64
 5   z                  577347 non-null  float64
 6   redshift           577347 non-null  float64
 7   spectral_type      577347 non-null  float64
 8   galaxy_population  577347 non-null  float64
 9   brightness         577347 non-null  float64
 10  u_norm             577347 non-null  float64
 11  g_norm             577347 non-null  float64
 12  r_norm             577347 non-null  float64
 13  i_norm             577347 non-null  float64
 14  z_norm             577347 non-null  float64
 15  luminosity         577347 non-null  float64
 16  u-g          

In [50]:
scores = [(np.mean(cv_scores_nn), 'StackingNN'), (np.mean(cv_scores_logistic), 'StackingLogistic'), (np.mean(cv_scores_ada), 'AdaBoost'), (np.mean(cv_scores_voting), 'VotingClassifier')]
scores.sort(key=lambda e: e[0], reverse=True)
best_model = scores[0][1]
if 'StackingNN' == best_model:
    nn_estimator = nn.Sequential(nn.Linear(90,30),nn.ELU(),nn.Linear(30,9), nn.ELU(), nn.Linear(9,3))
    model = MyStackingModel(models_5, ohe, True, nn_estimator, {'batch_size' : 32, 'lr' : 1e-4, 'max_epochs' : 100, 'model_name' : 'NN'})
elif 'StackingLogistic' == best_model:
    model = MyStackingModel(models_5, ohe, False, LogisticRegression(), {'model_name' : 'logistic'})
elif 'AdaBoost' == best_model:
    model = AdaBoostClassifier(estimator=MyVotingClassifier(models_1, ohe), n_estimators=5)
elif 'VotingClassifier' == best_model:
    model = MyVotingClassifier(models_5, ohe)
else:
    print('Unrecognized best model')
model.fit(X.values,y_labels)

Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/xgboost/core.py:748: FutureWarning: Pass `sample_weight` as keyword args.
  warnings.warn(msg, FutureWarning)


Model with name: XGBClassifier, has been fitted
Model with name: LightGBM, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: NN, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted


...

Model with name: Logistic, has been fitted
Model with name: RandomForest, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/xgboost/core.py:748: FutureWarning: Pass `sample_weight` as keyword args.
  warnings.warn(msg, FutureWarning)


Model with name: XGBClassifier, has been fitted
Model with name: LightGBM, has been fitted
Model with name: AdaBoost, has been fitted
Model with name: NN, has been fitted


/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",MyVotingClass...output=False))
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",5
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",1.0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](3,)","[0,1,2]"
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,MyVotingClassifier,MyVotingClass...output=False))
estimator_errors_ estimator_errors_: ndarray of floatsClassification error for each estimator in the boostedensemble.,"ndarray[float64](5,)","[0.03,0.24,0.37,0.49,0.56]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](5,)","[4.04,1.84,1.24,0.74,0.46]"
estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators.,list,"[MyVotingClass...output=False)), MyVotingClass...output=False)), MyVotingClass...output=False)), MyVotingClass...output=False)), ...]"
n_classes_ n_classes_: intThe number of classes.,int,3


In [51]:
df_test = pd.read_csv('/home/kyburg/kaggle/input/predicting_stellar_class/test.csv')
X_test = df_test.drop(columns=['id'])
X_test = preprocessing(X_test)
X_test = scaler.transform(X_test)
y_pred_test = model.predict(X_test)
y_pred_test = le.inverse_transform(y_pred_test)
submission = pd.DataFrame({
    'id' : df_test['id'],
    'Class' : y_pred_test
})
submission.to_csv('submission.csv', index=False)

/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but AdaBoostClassifier was fitted without feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kyburg/Code/Python/ML_env_312/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, 

In [ ]:
model_nn = NNClassifier(topology=[48,24,12], alpha=0.3)
train_loader = DataLoader(dataset=MyDataset(X.values,y), batch_size=32)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model_nn.parameters(), lr=1e-4)

In [ ]:
for epoch in range(100):
    print(f'Starting epoch: {epoch+1}/100')
    for x_batch,y_batch,_ in train_loader:
        optimizer.zero_grad()
        y_pred = model_nn(x_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
with torch.no_grad():
    y_pred_test = torch.argmax(model_nn(torch.from_numpy(X_test.values.astype(np.float32))), dim=1).numpy()
y_pred_test = le.inverse_transform(y_pred_test)
submission = pd.DataFrame({
    'id' : df_test['id'],
    'Class' : y_pred_test
})
submission.to_csv('submission.csv', index=False)

Starting epoch: 1/100
Starting epoch: 2/100
Starting epoch: 3/100
Starting epoch: 4/100
Starting epoch: 5/100
Starting epoch: 6/100
Starting epoch: 7/100
Starting epoch: 8/100
Starting epoch: 9/100
Starting epoch: 10/100
Starting epoch: 11/100
Starting epoch: 12/100
Starting epoch: 13/100
Starting epoch: 14/100
Starting epoch: 15/100
Starting epoch: 16/100
Starting epoch: 17/100
Starting epoch: 18/100
Starting epoch: 19/100
Starting epoch: 20/100
Starting epoch: 21/100
Starting epoch: 22/100
Starting epoch: 23/100
Starting epoch: 24/100
Starting epoch: 25/100
Starting epoch: 26/100
Starting epoch: 27/100
Starting epoch: 28/100
Starting epoch: 29/100
Starting epoch: 30/100
Starting epoch: 31/100
Starting epoch: 32/100
Starting epoch: 33/100
Starting epoch: 34/100
Starting epoch: 35/100
Starting epoch: 36/100
Starting epoch: 37/100
Starting epoch: 38/100
Starting epoch: 39/100
Starting epoch: 40/100
Starting epoch: 41/100
Starting epoch: 42/100
Starting epoch: 43/100
Starting epoch: 44/1